# QPSK-like CC Matrix Generation with Blind Rs/CFO

This notebook generates 11x24 CC-matrix `.pt` files for QPSK-like signals only (`QPSK`, `16QAM`, `64QAM`, `256QAM`). It reads the compact BOI+CFO-shift FFT vector payloads in `./2018`, estimates Rs from the length-20 `rs20` vector using the strict single-signal hand-written rule, estimates residual CFO from `cfo1`, and builds CC matrices from `x_processed` plus BOI inband power.

Runtime dependencies are local `pre_*.py` modules in this directory:

- `pre_cc_matrix_core_from_old.py`: CC helpers extracted from `CC_Matrix_generate.ipynb`; this notebook no longer executes or imports the old notebook.
- `pre_qpsklike_rs_cfo_estimator.py`: `rs20/cfo1` blind Rs/CFO estimation logic. The Rs estimator is the strict local hand-written rule from `pre_Rs-CFO-estimation-QPSKlike-pkl-strict-hand-unsupervised-more-features.ipynb`; it does not use truth, SNR, signal subtype, global `candidate_bins`, or dataset-level statistics.

Input PT contract enforced by this notebook:

- `sample["x_processed"]`: BOI+CFO-shifted signal used for CC matrix generation.
- `sample["vectors"]["rs20"]`: length-20 vector used for strict local Rs estimation.
- `sample["vectors"]["cfo1"]`: length-1 vector used for CFO estimation, with feature name exactly `["x4_top1_f"]`.
- `sample["vectors"]["lr445"]`: validated as present from the compact-vector payload used by the QPSK-like flow; not used by the Rs/CFO estimator.

A separate validation section scans both `./2018` and `./2022`, reads `rs20/cfo1` for every QPSK-like sample, skips 2018 negative-SNR samples, and reports Rs/CFO accuracy. That validation is only a proof that the estimator works; its results are not cached or reused by CC matrix generation.

For 2018 QPSK-like rows with negative truth SNR, the row is skipped for computation. In CC output generation it is kept as an error marker (`status=False`, `error="SKIP_NEGATIVE_SNR"`) and no CC matrix is computed for that index.


In [ ]:
from pathlib import Path
import sys


def _find_helper_project_dir(start=Path.cwd()):
    start = Path(start).resolve()
    search_roots = [start, *start.parents]
    for root in search_roots:
        if (root / "helper_function" / "__init__.py").exists():
            return root
    for root in search_roots[:3]:
        matches = list(root.glob("**/boi_shift_fft3_cfpattern_spl/helper_function/__init__.py"))
        if matches:
            return matches[0].parent.parent
    raise FileNotFoundError("Could not find helper_function from the current Jupyter working directory.")


PROJECT_DIR = _find_helper_project_dir()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import gc
import time

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

import helper_function.pre_cc_matrix_core_from_old as cc
import helper_function.pre_qpsklike_rs_cfo_estimator as rc

BASE = PROJECT_DIR

CFG = {
    "INPUT_ROOT": BASE,
    "OUT_ROOT": BASE / "qpsklike_cc_matrix_11x24_blind_rs_cfo_pt_dataset",
    "YEARS": ("2018",),
    "VALIDATION_YEARS": ("2018", "2022"),
    "N_BATCHES": 28,
    "N_PER_BATCH": 4000,
    "SKIP_EXISTING": False,
    "TARGET_SIGNAL_TYPES": {"qpsk", "16qam", "64qam", "256qam"},
    "DROP_2018_NEGATIVE_SNR": True,
    "NEGATIVE_SNR_ERROR": rc.NEGATIVE_SNR_ERROR,
    "RS_ABS_MIN_COUNT": 0,
    "RS_SCHEME_NAME": "rs20_strict_single_signal_hand_rule",
    "LR_VECTOR_KEY": "lr445",
    "RS_VECTOR_KEY": "rs20",
    "CFO_VECTOR_KEY": "cfo1",
    "CC_MAX_K": 5,
    "CC_REMOVE_MEAN": False,
    "ENABLE_WARPING_THEORY": True,
    "ENABLE_SCALING_THEORY": True,
    "CC_SCALE_THEORY": False,
    "CC_THEORY_USE_CFO_IN_PULSE_ALPHA": False,
    "CC_THEORY_SPAN_SYMBOLS": 12,
    "CC_THEORY_PULSE_SPS": 128,
    "ENABLE_WARPING_ESTIMATE": True,
    "ENABLE_SCALING_ESTIMATE": False,
    "EPS": 1e-12,
    "USE_CUDA": torch.cuda.is_available(),
    "CUDA_DEVICE": "cuda:0",
    "CUDA_COMPLEX_DTYPE": "complex128",
}

CFG["OUT_ROOT"] = Path(CFG["OUT_ROOT"])
CFG["INPUT_ROOT"] = Path(CFG["INPUT_ROOT"])

print("BASE:", BASE)
print("INPUT_ROOT:", CFG["INPUT_ROOT"])
print("OUT_ROOT:", CFG["OUT_ROOT"])
print("USE_CUDA:", CFG["USE_CUDA"], "device:", CFG["CUDA_DEVICE"])
print("CC helper module:", cc.__file__)
print("Rs/CFO estimator module:", rc.__file__)


BASE: /home/ruilin/SPL_Surreal/Surreal_source/chad-reproduce/chad-reproduce-sensors/BOI+SSCA+CC-Blind-generateselectedBOI+SSCA-metadata/BOI+FFT-2018+2022-generate/boi_shift_fft3_cfpattern_finalresult
INPUT_ROOT: /home/ruilin/SPL_Surreal/Surreal_source/chad-reproduce/chad-reproduce-sensors/BOI+SSCA+CC-Blind-generateselectedBOI+SSCA-metadata/BOI+FFT-2018+2022-generate/boi_shift_fft3_cfpattern_finalresult
OUT_ROOT: /home/ruilin/SPL_Surreal/Surreal_source/chad-reproduce/chad-reproduce-sensors/BOI+SSCA+CC-Blind-generateselectedBOI+SSCA-metadata/BOI+FFT-2018+2022-generate/boi_shift_fft3_cfpattern_finalresult/qpsklike_cc_matrix_11x24_blind_rs_cfo_pt_dataset
USE_CUDA: True device: cuda:0
CC helper module: /home/ruilin/SPL_Surreal/Surreal_source/chad-reproduce/chad-reproduce-sensors/BOI+SSCA+CC-Blind-generateselectedBOI+SSCA-metadata/BOI+FFT-2018+2022-generate/boi_shift_fft3_cfpattern_finalresult/pre_cc_matrix_core_from_old.py
Rs/CFO estimator module: /home/ruilin/SPL_Surreal/Surreal_source/cha

/home/ruilin/miniconda3/envs/fpt_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cc.configure_cc_runtime(
    USE_CUDA=bool(CFG["USE_CUDA"]),
    CUDA_DEVICE=CFG["CUDA_DEVICE"],
    CUDA_COMPLEX_DTYPE=CFG["CUDA_COMPLEX_DTYPE"],
)

print("CC runtime:", cc.CC_RUNTIME_CFG)
print("CC matrix shape:", (len(cc.CC_K_LIST), len(cc.CC_NM)))
print("CC estimate postprocess: warping=", CFG["ENABLE_WARPING_ESTIMATE"], "scaling=", CFG["ENABLE_SCALING_ESTIMATE"])
print("CC theory postprocess: warping=", CFG["ENABLE_WARPING_THEORY"], "scaling=", CFG["ENABLE_SCALING_THEORY"], "scale_theory=", CFG["CC_SCALE_THEORY"])


CC runtime: {'USE_CUDA': True, 'CUDA_DEVICE': 'cuda:0', 'CUDA_COMPLEX_DTYPE': 'complex128'}
CC matrix shape: (11, 24)
CC estimate postprocess: warping= True scaling= False
CC theory postprocess: warping= True scaling= True scale_theory= False


In [3]:
def pt_path_for_batch(year, batch_id):
    return Path(CFG["INPUT_ROOT"]) / str(year) / f"{year}_batch_{int(batch_id):02d}_boi_shift_fft3_compact_vectors.pt"


def out_path_for_batch(year, batch_id):
    return Path(CFG["OUT_ROOT"]) / f"{year}_qpsklike_cc_11x24_blindrs_cfo_batch_{int(batch_id):02d}.pt"


def load_pack(path):
    return rc.load_pack(path)


def is_target_sample(sample):
    return rc.is_qpsklike_sample(sample, CFG["TARGET_SIGNAL_TYPES"])


def is_negative_snr_error(sample, year):
    return rc.is_negative_snr_skip(sample, year, CFG["DROP_2018_NEGATIVE_SNR"])


def sample_signal_type(sample):
    return rc.sample_signal_type(sample)


def sample_snr(sample):
    return rc.sample_snr(sample)


def validate_compact_pack_schema(pack):
    return rc.validate_compact_pack_schema(
        pack,
        rs_vector_key=CFG["RS_VECTOR_KEY"],
        cfo_vector_key=CFG["CFO_VECTOR_KEY"],
        lr_vector_key=CFG["LR_VECTOR_KEY"],
        require_lr_vector=True,
    )


def validate_vector_names(pack):
    schema = validate_compact_pack_schema(pack)
    return schema["rs_names"], schema["cfo_names"]


In [4]:
def collect_absx2_candidate_bins(cfg=CFG):
    if cfg["RS_VECTOR_KEY"] == "rs20":
        info = {
            "candidate_bins": [],
            "candidate_bin_count": 0,
            "candidate_min_count": 0,
            "scan_summary": pd.DataFrame(),
            "top_bin_counts": pd.DataFrame(columns=["bin", "count"]),
        }
        print("rs20 strict local estimator: no global candidate bins are used")
        print("candidate_bin_count:", info["candidate_bin_count"])
        return info

    info = rc.collect_absx2_candidate_bins(
        input_root=cfg["INPUT_ROOT"],
        years=cfg["YEARS"],
        n_batches=cfg["N_BATCHES"],
        min_count=cfg["RS_ABS_MIN_COUNT"],
        rs_vector_key=cfg["RS_VECTOR_KEY"],
        cfo_vector_key=cfg["CFO_VECTOR_KEY"],
        target_signal_types=cfg["TARGET_SIGNAL_TYPES"],
        drop_2018_negative_snr=cfg["DROP_2018_NEGATIVE_SNR"],
    )
    print("candidate_bin_count:", info["candidate_bin_count"])
    return info


## Rs20/CFO Estimator Validation

This section scans all compact PT files in `./2018` and `./2022`, reads only each QPSK-like sample's local `rs20` and `cfo1` vectors, and computes Rs/CFO estimates through `pre_qpsklike_rs_cfo_estimator.py`. It skips 2018 negative-SNR samples.

This is only an estimator sanity check. The later CC-matrix generation still calls the estimator independently for each selected QPSK-like sample inside `build_cc_for_sample()`.


In [5]:
def truth_rs_for_sample(sample):
    truth = sample.get("truth", {})
    return float(truth.get("symbol_rate", truth.get("rs_truth", np.nan)))


def truth_cfo_shifted_for_sample(sample):
    truth = sample.get("truth", {})
    preprocess = sample.get("preprocess", {})
    return float(truth.get("cfo_truth_shifted", preprocess.get("truth_guide_cfo", np.nan)))


def validate_rs20_cfo_estimator_all_years(cfg=CFG):
    if cfg["RS_VECTOR_KEY"] != "rs20":
        raise ValueError("This validation cell is intended for RS_VECTOR_KEY='rs20'.")

    rows = []
    skipped_rows = []
    t0 = time.time()
    candidate_bins = set()  # rs20 strict local estimator does not use candidate bins.

    for year in cfg["VALIDATION_YEARS"]:
        for batch_id in range(1, int(cfg["N_BATCHES"]) + 1):
            path = pt_path_for_batch(year, batch_id)
            if not path.exists():
                raise FileNotFoundError(path)

            pack = load_pack(path)
            schema = validate_compact_pack_schema(pack)
            rs_names, cfo_names = schema["rs_names"], schema["cfo_names"]

            n_qpsklike = 0
            n_skipped_2018_negative_snr = 0
            n_estimator_error = 0

            for sample in pack["samples"]:
                if not is_target_sample(sample):
                    continue
                n_qpsklike += 1

                if is_negative_snr_error(sample, year):
                    n_skipped_2018_negative_snr += 1
                    skipped_rows.append({
                        "year": str(year),
                        "batch_id": int(batch_id),
                        "signal_index": int(sample.get("meta", {}).get("signal_index", sample.get("meta", {}).get("global_index", -1))),
                        "reason": CFG["NEGATIVE_SNR_ERROR"],
                    })
                    continue

                rs_truth = truth_rs_for_sample(sample)
                cfo_truth = truth_cfo_shifted_for_sample(sample)
                try:
                    est = rc.estimate_rs_cfo_from_sample(
                        sample,
                        rs_names,
                        cfo_names,
                        candidate_bins,
                        rs_vector_key=cfg["RS_VECTOR_KEY"],
                        cfo_vector_key=cfg["CFO_VECTOR_KEY"],
                    )
                    rs_est = float(est["rs_est"])
                    cfo_est = float(est["cfo_est"])
                    rs_source = str(est["rs_source"])
                    error = ""
                except Exception as exc:
                    n_estimator_error += 1
                    rs_est = np.nan
                    cfo_est = np.nan
                    rs_source = "ERROR"
                    error = repr(exc)

                rows.append({
                    "year": str(year),
                    "batch_id": int(batch_id),
                    "signal_index": int(sample.get("meta", {}).get("signal_index", sample.get("meta", {}).get("global_index", -1))),
                    "signal_type": sample_signal_type(sample),
                    "snr": sample_snr(sample),
                    "rs_truth": rs_truth,
                    "rs_est": rs_est,
                    "rs_err": abs(rs_est - rs_truth) if np.isfinite(rs_est) and np.isfinite(rs_truth) else np.nan,
                    "rs_ok_1bin": bool(np.isfinite(rs_est) and np.isfinite(rs_truth) and abs(rs_est - rs_truth) <= rc.BIN),
                    "cfo_truth_shifted": cfo_truth,
                    "cfo_est": cfo_est,
                    "cfo_err": abs(float(rc.wrap_freq_scalar(cfo_est - cfo_truth))) if np.isfinite(cfo_est) and np.isfinite(cfo_truth) else np.nan,
                    "cfo_ok_1x4bin": bool(np.isfinite(cfo_est) and np.isfinite(cfo_truth) and abs(float(rc.wrap_freq_scalar(cfo_est - cfo_truth))) <= rc.BIN / 4.0),
                    "rs_source": rs_source,
                    "error": error,
                })

            print(
                f"validation {year} batch {batch_id:02d}: "
                f"qpsklike={n_qpsklike} skipped_2018_negative_snr={n_skipped_2018_negative_snr} "
                f"estimator_errors={n_estimator_error} elapsed={time.time() - t0:.1f}s",
                flush=True,
            )
            del pack
            gc.collect()

    pred = pd.DataFrame(rows)
    skipped = pd.DataFrame(skipped_rows)
    if pred.empty:
        raise RuntimeError("No validation rows were collected.")

    def summarize_group(g, group_name):
        return {
            "group": str(group_name),
            "n": int(len(g)),
            "rs_acc_percent": float(100.0 * g["rs_ok_1bin"].mean()),
            "rs_fails": int((~g["rs_ok_1bin"]).sum()),
            "rs_mae": float(g["rs_err"].mean()),
            "rs_p95": float(g["rs_err"].quantile(0.95)),
            "cfo_acc_percent": float(100.0 * g["cfo_ok_1x4bin"].mean()),
            "cfo_fails": int((~g["cfo_ok_1x4bin"]).sum()),
            "cfo_mae": float(g["cfo_err"].mean()),
            "estimator_errors": int((g["error"].astype(str) != "").sum()),
        }

    summary_rows = [summarize_group(pred, "ALL")]
    summary_rows += [summarize_group(g, year) for year, g in pred.groupby("year", sort=True)]
    summary_rows += [summarize_group(g, sig) for sig, g in pred.groupby("signal_type", sort=True)]
    summary = pd.DataFrame(summary_rows)

    by_year_signal = (
        pred.groupby(["year", "signal_type"], as_index=False)
        .agg(
            n=("rs_ok_1bin", "size"),
            rs_acc_percent=("rs_ok_1bin", lambda x: 100.0 * float(x.mean())),
            rs_fails=("rs_ok_1bin", lambda x: int((~x).sum())),
            cfo_acc_percent=("cfo_ok_1x4bin", lambda x: 100.0 * float(x.mean())),
            cfo_fails=("cfo_ok_1x4bin", lambda x: int((~x).sum())),
        )
    )

    source_summary = (
        pred.groupby("rs_source", as_index=False)
        .agg(
            n=("rs_ok_1bin", "size"),
            rs_acc_percent=("rs_ok_1bin", lambda x: 100.0 * float(x.mean())),
            rs_fails=("rs_ok_1bin", lambda x: int((~x).sum())),
        )
        .sort_values("n", ascending=False)
    )

    print("\nEstimator contract:")
    print({
        "rs_vector_key": cfg["RS_VECTOR_KEY"],
        "cfo_vector_key": cfg["CFO_VECTOR_KEY"],
        "single_signal_unsupervised": True,
        "uses_truth_for_estimation": False,
        "uses_global_candidate_bins": False,
        "uses_dataset_statistics_for_estimation": False,
        "skips_2018_negative_snr": True,
    })
    print("\nValidation summary")
    print(summary.to_string(index=False))
    print("\nValidation by year/signal_type")
    print(by_year_signal.to_string(index=False))
    print("\nRs source summary")
    print(source_summary.to_string(index=False))
    print("\nSkipped 2018 negative-SNR rows:", len(skipped))

    return pred, summary, by_year_signal, source_summary, skipped


Run the next cell to validate the estimator on both years before generating CC matrices. The returned `rs20_validation_pred` is not consumed by the CC generation cells.


In [6]:
rs20_validation_pred, rs20_validation_summary, rs20_validation_by_year_signal, rs20_validation_source_summary, rs20_validation_skipped = validate_rs20_cfo_estimator_all_years(CFG)


validation 2018 batch 01: qpsklike=1996 skipped_2018_negative_snr=0 estimator_errors=0 elapsed=11.6s
validation 2018 batch 02: qpsklike=1992 skipped_2018_negative_snr=0 estimator_errors=0 elapsed=22.3s
validation 2018 batch 03: qpsklike=1992 skipped_2018_negative_snr=0 estimator_errors=0 elapsed=33.7s
validation 2018 batch 04: qpsklike=1992 skipped_2018_negative_snr=0 estimator_errors=0 elapsed=44.5s
validation 2018 batch 05: qpsklike=1996 skipped_2018_negative_snr=0 estimator_errors=0 elapsed=54.9s
validation 2018 batch 06: qpsklike=1992 skipped_2018_negative_snr=0 estimator_errors=0 elapsed=64.9s
validation 2018 batch 07: qpsklike=1992 skipped_2018_negative_snr=0 estimator_errors=0 elapsed=74.9s
validation 2018 batch 08: qpsklike=1992 skipped_2018_negative_snr=0 estimator_errors=0 elapsed=84.6s
validation 2018 batch 09: qpsklike=1996 skipped_2018_negative_snr=0 estimator_errors=0 elapsed=94.2s
validation 2018 batch 10: qpsklike=1992 skipped_2018_negative_snr=0 estimator_errors=0 elap

In [7]:
def get_power_for_cc(sample):
    power = sample.get("power", {})
    for key in ["p_boi_inband_signal", "P_x_processed", "P_x_bpf", "P_total"]:
        if key in power:
            val = float(power[key])
            if np.isfinite(val) and val > 0.0:
                return val, key
    x = np.asarray(sample["x_processed"])
    val = float(np.mean(np.abs(x) ** 2))
    if not np.isfinite(val) or val <= 0.0:
        raise ValueError(f"Invalid power for CC scaling: {val}")
    return val, "mean_abs_x_processed_sq"


def build_cc_for_sample(sample, rs_names, cfo_names, candidate_bins):
    # CC generation intentionally estimates Rs/CFO independently per sample.
    # The validation section above is not cached or reused here.
    est = rc.estimate_rs_cfo_from_sample(
        sample,
        rs_names,
        cfo_names,
        candidate_bins,
        rs_vector_key=CFG["RS_VECTOR_KEY"],
        cfo_vector_key=CFG["CFO_VECTOR_KEY"],
    )
    power_scale, power_source = get_power_for_cc(sample)
    sig = sample_signal_type(sample)
    pdf_name = cc.CC_PDF_MAP.get(sig, "QPSK")
    pattern_dict = cc.CC_PATTERN_MAP.get(sig, cc.qpsk)
    beta = float(sample.get("truth", {}).get("excess_bw", np.nan))

    theory_raw = cc.build_theoretical_cc_matrix_from_table3_pattern(
        pdf_name,
        pattern_dict,
        T0=est["T0_est"],
        f0=est["cfo_est"],
        beta=beta,
        span_symbols=int(CFG["CC_THEORY_SPAN_SYMBOLS"]),
        pulse_sps=int(CFG["CC_THEORY_PULSE_SPS"]),
        use_cfo_in_pulse_alpha=bool(CFG["CC_THEORY_USE_CFO_IN_PULSE_ALPHA"]),
    )
    theory_proc, theory_abs = cc.apply_cc_postprocessing_with_power(
        theory_raw["matrix_complex"],
        power_scale,
        cc.CC_NM,
        enable_warping=bool(CFG["ENABLE_WARPING_THEORY"]),
        enable_scaling=bool(CFG["ENABLE_SCALING_THEORY"]),
        is_theory=True,
        scale_theory=bool(CFG["CC_SCALE_THEORY"]),
        eps=float(CFG["EPS"]),
    )

    raw = cc.build_estimated_cc_matrix_11x24_from_signal_pattern(
        np.asarray(sample["x_processed"], dtype=np.complex128),
        est["T0_est"],
        est["cfo_est"],
        pattern_dict,
        remove_mean=bool(CFG["CC_REMOVE_MEAN"]),
        max_k=int(CFG["CC_MAX_K"]),
    )
    proc, proc_abs = cc.apply_cc_postprocessing_with_power(
        raw["matrix_complex"],
        power_scale,
        cc.CC_NM,
        enable_warping=bool(CFG["ENABLE_WARPING_ESTIMATE"]),
        enable_scaling=bool(CFG["ENABLE_SCALING_ESTIMATE"]),
        is_theory=False,
        eps=float(CFG["EPS"]),
    )
    return theory_raw, theory_proc, theory_abs, raw, proc, proc_abs, est, power_scale, power_source


def make_empty_outputs(n):
    shape = (int(n), len(cc.CC_K_LIST), len(cc.CC_NM))
    nan_complex = np.complex64(np.nan + 1j * np.nan)
    return {
        "estimate_complex": np.full(shape, nan_complex, dtype=np.complex64),
        "estimate_abs": np.full(shape, np.nan, dtype=np.float32),
        "theory_complex": np.full(shape, nan_complex, dtype=np.complex64),
        "theory_abs": np.full(shape, np.nan, dtype=np.float32),
        "signal_index": np.zeros(n, dtype=np.int64),
        "local_index": np.zeros(n, dtype=np.int64),
        "T0": np.full(n, np.nan, dtype=np.float64),
        "CFO_residual": np.full(n, np.nan, dtype=np.float64),
        "rs_est": np.full(n, np.nan, dtype=np.float64),
        "cfo_est": np.full(n, np.nan, dtype=np.float64),
        "truth_rs_for_debug": np.full(n, np.nan, dtype=np.float64),
        "truth_cfo_shifted_for_debug": np.full(n, np.nan, dtype=np.float64),
        "snr": np.full(n, np.nan, dtype=np.float64),
        "estimated_BOI_center": np.full(n, np.nan, dtype=np.float64),
        "boi_inband_signal": np.full(n, np.nan, dtype=np.float64),
        "boi_inband_total": np.full(n, np.nan, dtype=np.float64),
        "processed_total": np.full(n, np.nan, dtype=np.float64),
        "original_total": np.full(n, np.nan, dtype=np.float64),
        "power_for_scaling": np.full(n, np.nan, dtype=np.float64),
        "beta": np.full(n, np.nan, dtype=np.float64),
        "status": np.zeros(n, dtype=bool),
        "error": [""] * n,
        "signal_type": [""] * n,
        "rs_source": [""] * n,
        "power_source": [""] * n,
        "pt_source": [""] * n,
    }


def fill_metadata(out, i, sample, src_name):
    meta = sample.get("meta", {})
    truth = sample.get("truth", {})
    preprocess = sample.get("preprocess", {})
    power = sample.get("power", {})

    out["signal_index"][i] = int(meta.get("signal_index", meta.get("global_index", -1)))
    out["local_index"][i] = int(meta.get("batch_local_index", -1)) + 1
    out["truth_rs_for_debug"][i] = float(truth.get("symbol_rate", truth.get("rs_truth", np.nan)))
    out["truth_cfo_shifted_for_debug"][i] = float(truth.get("cfo_truth_shifted", preprocess.get("truth_guide_cfo", np.nan)))
    out["snr"][i] = sample_snr(sample)
    out["estimated_BOI_center"][i] = float(preprocess.get("estimated_boi_center_hz", preprocess.get("center_shift_hz", np.nan)))
    out["boi_inband_signal"][i] = float(power.get("p_boi_inband_signal", np.nan))
    out["boi_inband_total"][i] = float(power.get("P_x_bpf", np.nan))
    out["processed_total"][i] = float(power.get("P_x_processed", np.nan))
    out["original_total"][i] = float(power.get("P_total", np.nan))
    out["beta"][i] = float(truth.get("excess_bw", np.nan))
    out["signal_type"][i] = str(meta.get("signal_type", "")).lower()
    out["pt_source"][i] = str(src_name)


def convert_outputs_to_torch(out):
    return {
        "estimate_complex": torch.from_numpy(out["estimate_complex"]),
        "estimate_abs": torch.from_numpy(out["estimate_abs"]),
        "theory_complex": torch.from_numpy(out["theory_complex"]),
        "theory_abs": torch.from_numpy(out["theory_abs"]),
        "signal_index": torch.from_numpy(out["signal_index"]),
        "local_index": torch.from_numpy(out["local_index"]),
        "T0": torch.from_numpy(out["T0"]),
        "CFO_residual": torch.from_numpy(out["CFO_residual"]),
        "rs_est": torch.from_numpy(out["rs_est"]),
        "cfo_est": torch.from_numpy(out["cfo_est"]),
        "truth_rs_for_debug": torch.from_numpy(out["truth_rs_for_debug"]),
        "truth_cfo_shifted_for_debug": torch.from_numpy(out["truth_cfo_shifted_for_debug"]),
        "snr": torch.from_numpy(out["snr"]),
        "estimated_BOI_center": torch.from_numpy(out["estimated_BOI_center"]),
        "boi_inband_signal": torch.from_numpy(out["boi_inband_signal"]),
        "boi_inband_total": torch.from_numpy(out["boi_inband_total"]),
        "processed_total": torch.from_numpy(out["processed_total"]),
        "original_total": torch.from_numpy(out["original_total"]),
        "power_for_scaling": torch.from_numpy(out["power_for_scaling"]),
        "beta": torch.from_numpy(out["beta"]),
        "status": torch.from_numpy(out["status"]),
        "error": out["error"],
        "signal_type": out["signal_type"],
        "rs_source": out["rs_source"],
        "power_source": out["power_source"],
        "pt_source": out["pt_source"],
        "k_list": list(cc.CC_K_LIST),
        "nm_list": list(cc.CC_NM),
        "col_labels": list(cc.CC_COL_LABELS),
    }


In [8]:
def generate_one_batch(year, batch_id, candidate_info, cfg=CFG):
    src_path = pt_path_for_batch(year, batch_id)
    dst_path = out_path_for_batch(year, batch_id)
    dst_path.parent.mkdir(parents=True, exist_ok=True)

    if bool(cfg["SKIP_EXISTING"]) and dst_path.exists() and dst_path.stat().st_size > 0:
        print(f"skip existing: {dst_path}")
        return dst_path

    pack = load_pack(src_path)
    input_schema = validate_compact_pack_schema(pack)
    rs_names, cfo_names = input_schema["rs_names"], input_schema["cfo_names"]
    selected = [s for s in pack["samples"] if is_target_sample(s)]
    out = make_empty_outputs(len(selected))
    candidate_bins = set(candidate_info["candidate_bins"])

    t0 = time.time()
    for i, sample in enumerate(tqdm(selected, desc=f"{year} batch {batch_id:02d} CC", leave=False)):
        fill_metadata(out, i, sample, src_path.name)
        try:
            if is_negative_snr_error(sample, year):
                out["error"][i] = cfg["NEGATIVE_SNR_ERROR"]
                out["status"][i] = False
                continue

            theory_raw, theory_proc, theory_abs, raw, proc, proc_abs, est, power_scale, power_source = build_cc_for_sample(
                sample,
                rs_names,
                cfo_names,
                candidate_bins,
            )
            out["estimate_complex"][i] = proc.astype(np.complex64, copy=False)
            out["estimate_abs"][i] = proc_abs.astype(np.float32, copy=False)
            out["theory_complex"][i] = theory_proc.astype(np.complex64, copy=False)
            out["theory_abs"][i] = theory_abs.astype(np.float32, copy=False)
            out["T0"][i] = est["T0_est"]
            out["CFO_residual"][i] = est["cfo_est"]
            out["rs_est"][i] = est["rs_est"]
            out["cfo_est"][i] = est["cfo_est"]
            out["power_for_scaling"][i] = power_scale
            out["rs_source"][i] = est["rs_source"]
            out["power_source"][i] = power_source
            out["status"][i] = True
        except Exception as exc:
            out["error"][i] = repr(exc)
            out["status"][i] = False

        if (i + 1) % 100 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    torch_pack = convert_outputs_to_torch(out)
    torch_pack["meta"] = {
        "version": "qpsklike_cc_11x24_blind_rs_cfo_from_compact_vectors_v2",
        "dataset_name": str(year),
        "batch_id": int(batch_id),
        "source_pt": str(src_path),
        "num_input_samples": int(len(pack["samples"])),
        "num_selected_qpsklike": int(len(selected)),
        "num_negative_snr_error_marked": int(sum(is_negative_snr_error(s, year) for s in selected)),
        "num_saved": int(len(selected)),
        "num_ok": int(out["status"].sum()),
        "num_error": int((~out["status"]).sum()),
        "negative_snr_error_marker": cfg["NEGATIVE_SNR_ERROR"],
        "drop_2018_negative_snr": bool(cfg["DROP_2018_NEGATIVE_SNR"]),
        "target_signal_types": sorted(cfg["TARGET_SIGNAL_TYPES"]),
        "blind_rs_cfo": True,
        "rs_scheme": cfg["RS_SCHEME_NAME"],
        "input_contract": {
            "x_processed": "sample['x_processed'] is read directly from compact PT and used for CC matrix generation",
            "lr445": "sample['vectors']['lr445'] is required and validated as present",
            "rs20": "sample['vectors']['rs20'] must have length 20 and is used for strict local hand-written Rs estimation",
            "cfo1": "sample['vectors']['cfo1'] must have feature_names ['x4_top1_f'] and is used for CFO estimation",
        },
        "lr_vector_key": cfg["LR_VECTOR_KEY"],
        "rs_vector_key": cfg["RS_VECTOR_KEY"],
        "cfo_vector_key": cfg["CFO_VECTOR_KEY"],
        "lr445_feature_count": int(len(input_schema["lr_names"])),
        "rs_feature_count": int(len(input_schema["rs_names"])),
        "rs_feature_names": list(input_schema["rs_names"]),
        "cfo1_feature_names": list(input_schema["cfo_names"]),
        "rs_candidate_bin_count": int(candidate_info["candidate_bin_count"]),
        "rs_candidate_min_count": int(candidate_info["candidate_min_count"]),
        "rs_estimator": "rs20 strict local hand-written rule from pre_Rs-CFO-estimation-QPSKlike-pkl-strict-hand-unsupervised-more-features.ipynb",
        "cfo_estimator": "wrap_freq(cfo1[0] / 4)",
        "cc_helper_module": str(Path(cc.__file__).name),
        "rs_cfo_estimator_module": str(Path(rc.__file__).name),
        "matrix_shape": [len(cc.CC_K_LIST), len(cc.CC_NM)],
        "cc_config": {
            "CC_MAX_K": int(cfg["CC_MAX_K"]),
            "CC_REMOVE_MEAN": bool(cfg["CC_REMOVE_MEAN"]),
            "ENABLE_WARPING_THEORY": bool(cfg["ENABLE_WARPING_THEORY"]),
            "ENABLE_SCALING_THEORY": bool(cfg["ENABLE_SCALING_THEORY"]),
            "CC_SCALE_THEORY": bool(cfg["CC_SCALE_THEORY"]),
            "CC_THEORY_USE_CFO_IN_PULSE_ALPHA": bool(cfg["CC_THEORY_USE_CFO_IN_PULSE_ALPHA"]),
            "CC_THEORY_SPAN_SYMBOLS": int(cfg["CC_THEORY_SPAN_SYMBOLS"]),
            "CC_THEORY_PULSE_SPS": int(cfg["CC_THEORY_PULSE_SPS"]),
            "ENABLE_WARPING_ESTIMATE": bool(cfg["ENABLE_WARPING_ESTIMATE"]),
            "ENABLE_SCALING_ESTIMATE": bool(cfg["ENABLE_SCALING_ESTIMATE"]),
            "CUDA_COMPLEX_DTYPE": cfg["CUDA_COMPLEX_DTYPE"],
            "power_source_priority": ["p_boi_inband_signal", "P_x_processed", "P_x_bpf", "P_total"],
        },
    }

    torch.save(torch_pack, dst_path)
    print(
        f"saved {dst_path.name}: selected={len(selected)} ok={torch_pack['meta']['num_ok']} "
        f"error={torch_pack['meta']['num_error']} neg_snr={torch_pack['meta']['num_negative_snr_error_marked']} "
        f"elapsed={time.time() - t0:.1f}s size={dst_path.stat().st_size / 1024 / 1024:.2f} MB",
        flush=True,
    )
    del pack, selected, out, torch_pack
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return dst_path


def generate_all(cfg=CFG):
    cfg["OUT_ROOT"].mkdir(parents=True, exist_ok=True)
    candidate_info = collect_absx2_candidate_bins(cfg)
    candidate_info["scan_summary"].to_csv(cfg["OUT_ROOT"] / "qpsklike_scan_summary.csv", index=False)
    candidate_info["top_bin_counts"].to_csv(cfg["OUT_ROOT"] / "rs_absx2_top_bin_counts.csv", index=False)

    paths = []
    for year in cfg["YEARS"]:
        for batch_id in range(1, int(cfg["N_BATCHES"]) + 1):
            paths.append(generate_one_batch(year, batch_id, candidate_info, cfg))

    print("done. files:", len(paths))
    print("output root:", cfg["OUT_ROOT"])
    return paths


In [9]:
# Optional quick inspection before the full run.
# Negative-SNR 2018 QPSK-like rows are kept as error-marker rows instead of being computed.
example = pt_path_for_batch(CFG["YEARS"][0], 1)
pack = load_pack(example)
schema = validate_compact_pack_schema(pack)
print("input schema:", {"lr445": len(schema["lr_names"]), CFG["RS_VECTOR_KEY"]: len(schema["rs_names"]), "cfo1": schema["cfo_names"], "validated_samples": schema["validated_sample_count"]})
selected_count = sum(is_target_sample(s) for s in pack["samples"])
print(example.name, "samples=", len(pack["samples"]), "selected_qpsklike=", selected_count)
del pack

example_2018 = pt_path_for_batch("2018", 1)
pack = load_pack(example_2018)
selected_2018 = [s for s in pack["samples"] if is_target_sample(s)]
neg_2018 = sum(is_negative_snr_error(s, "2018") for s in selected_2018)
print(example_2018.name, "selected_qpsklike=", len(selected_2018), "negative_snr_error_marked=", neg_2018)
del pack


input schema: {'lr445': 445, 'rs20': 20, 'cfo1': ['x4_top1_f'], 'validated_samples': 3992}
2018_batch_01_boi_shift_fft3_compact_vectors.pt samples= 3992 selected_qpsklike= 1996
2018_batch_01_boi_shift_fft3_compact_vectors.pt selected_qpsklike= 1996 negative_snr_error_marked= 0


In [10]:
# Run this cell to generate the CC output .pt files for CFG["YEARS"].
# It may take a long time because every selected QPSK-like x_processed signal is converted to an 11x24 CC matrix.
paths = generate_all(CFG)


rs20 strict local estimator: no global candidate bins are used
candidate_bin_count: 0


saved 2018_qpsklike_cc_11x24_blindrs_cfo_batch_01.pt: selected=1996 ok=1996 error=0 neg_snr=0 elapsed=2538.1s size=12.39 MB


saved 2018_qpsklike_cc_11x24_blindrs_cfo_batch_02.pt: selected=1992 ok=1992 error=0 neg_snr=0 elapsed=2603.3s size=12.36 MB


saved 2018_qpsklike_cc_11x24_blindrs_cfo_batch_03.pt: selected=1992 ok=1992 error=0 neg_snr=0 elapsed=2846.8s size=12.36 MB


saved 2018_qpsklike_cc_11x24_blindrs_cfo_batch_04.pt: selected=1992 ok=1992 error=0 neg_snr=0 elapsed=2590.5s size=12.36 MB


...

saved 2018_qpsklike_cc_11x24_blindrs_cfo_batch_24.pt: selected=1992 ok=1992 error=0 neg_snr=0 elapsed=2572.5s size=12.36 MB


saved 2018_qpsklike_cc_11x24_blindrs_cfo_batch_25.pt: selected=1996 ok=1996 error=0 neg_snr=0 elapsed=2576.7s size=12.39 MB


saved 2018_qpsklike_cc_11x24_blindrs_cfo_batch_26.pt: selected=1996 ok=1996 error=0 neg_snr=0 elapsed=2579.9s size=12.39 MB


saved 2018_qpsklike_cc_11x24_blindrs_cfo_batch_27.pt: selected=1992 ok=1992 error=0 neg_snr=0 elapsed=2573.9s size=12.36 MB


saved 2018_qpsklike_cc_11x24_blindrs_cfo_batch_28.pt: selected=1996 ok=1996 error=0 neg_snr=0 elapsed=2596.7s size=12.39 MB
done. files: 28
output root: /home/ruilin/SPL_Surreal/Surreal_source/chad-reproduce/chad-reproduce-sensors/BOI+SSCA+CC-Blind-generateselectedBOI+SSCA-metadata/BOI+FFT-2018+2022-generate/boi_shift_fft3_cfpattern_finalresult/qpsklike_cc_matrix_11x24_blind_rs_cfo_pt_dataset
